# AI Investment Landscape — Market Returns

This notebook turns the project's 45-company AI universe into a reproducible first-pass market study.

**Three layers**
1. Models, assistants, and AI software.
2. Core AI technology: chips, foundries, memory, networking, and servers.
3. Second-order beneficiaries: power, cooling, electrical infrastructure, construction, optics, and connectors.

The main question is not simply *which stocks went up?* It is whether looking at the **full AI dependency chain** reveals less-obvious beneficiaries that a normal "AI stock" screen misses.

This notebook is descriptive research, not investment advice. Historical returns do not establish that AI caused those returns.

## 1. Setup and company universe

The notebook reads `../data/company_universe.csv` rather than hard-coding the company list. Private companies such as OpenAI remain in the industry map but are automatically excluded from stock-return calculations.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import yfinance as yf

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

candidates = [
    Path("../data/company_universe.csv"),
    Path("projects/ai_investment_landscape/data/company_universe.csv"),
    Path("data/company_universe.csv"),
]
UNIVERSE_PATH = next((p for p in candidates if p.exists()), None)
if UNIVERSE_PATH is None:
    raise FileNotFoundError("Could not locate company_universe.csv")

universe = pd.read_csv(UNIVERSE_PATH)
universe["non_obviousness_score"] = universe["non_obviousness"].map(
    {"Low": 1, "Medium": 2, "High": 3, "Very High": 4}
)

public = universe.loc[
    universe["public_status"].str.lower().eq("public")
    & universe["ticker"].notna()
].copy()

print(f"Universe: {len(universe)} companies")
print(f"Public-equity sample: {len(public)} companies")
display(universe.groupby(["tier", "tier_name", "public_status"]).size().rename("companies").reset_index())
display(universe.loc[universe["public_status"].str.lower().ne("public"), ["company", "primary_ai_link"]])

## 2. Download adjusted prices

`yfinance` is used for exploratory market data and is already part of the repository requirements. We request a ten-year window plus a small buffer and use adjusted prices. A production-grade study should later validate market data and corporate actions against a formal data source.

In [ ]:
BENCHMARK = "SPY"
HORIZONS = [1, 3, 5, 10]

end_date = pd.Timestamp.today().normalize()
start_date = end_date - pd.DateOffset(years=10) - pd.Timedelta(days=14)

tickers = sorted(public["ticker"].unique())
symbols = tickers + [BENCHMARK]

raw = yf.download(
    symbols,
    start=start_date.date().isoformat(),
    end=(end_date + pd.Timedelta(days=1)).date().isoformat(),
    auto_adjust=True,
    progress=False,
    threads=True,
)

prices = raw["Close"].copy() if isinstance(raw.columns, pd.MultiIndex) else raw[["Close"]]
prices = prices.sort_index().dropna(how="all")

coverage = pd.DataFrame({
    "ticker": prices.columns,
    "first_date": [prices[c].first_valid_index() for c in prices.columns],
    "last_date": [prices[c].last_valid_index() for c in prices.columns],
    "observations": [prices[c].notna().sum() for c in prices.columns],
})
display(coverage.sort_values("first_date"))

## 3. Calculate comparable trailing returns

A company receives a 10-year return only if it actually has approximately ten years of trading history. Newer listings are left blank rather than silently substituting a shorter period.

In [ ]:
def trailing_stats(series, years, tolerance_days=10):
    s = series.dropna().sort_index()
    if s.empty:
        return None

    end_actual = s.index.max()
    target = end_actual - pd.DateOffset(years=years)
    eligible = s.loc[target:]
    if eligible.empty:
        return None

    start_actual = eligible.index.min()
    if start_actual > target + pd.Timedelta(days=tolerance_days):
        return None

    start_price, end_price = float(s.loc[start_actual]), float(s.loc[end_actual])
    actual_years = (end_actual - start_actual).days / 365.25
    total_return = end_price / start_price - 1
    cagr = (end_price / start_price) ** (1 / actual_years) - 1

    return total_return, cagr


def available_stats(series):
    s = series.dropna().sort_index()
    if len(s) < 2:
        return np.nan, np.nan, np.nan
    years = (s.index.max() - s.index.min()).days / 365.25
    total_return = float(s.iloc[-1] / s.iloc[0] - 1)
    cagr = (float(s.iloc[-1] / s.iloc[0])) ** (1 / years) - 1
    return years, total_return, cagr


benchmark = {
    y: trailing_stats(prices[BENCHMARK], y)[0]
    for y in HORIZONS
    if trailing_stats(prices[BENCHMARK], y)
}

rows = []
for _, company in public.iterrows():
    t = company["ticker"]
    if t not in prices:
        continue

    history_years, available_return, available_cagr = available_stats(prices[t])
    row = {
        "company": company["company"],
        "ticker": t,
        "tier": company["tier"],
        "tier_name": company["tier_name"],
        "primary_ai_link": company["primary_ai_link"],
        "non_obviousness": company["non_obviousness"],
        "non_obviousness_score": company["non_obviousness_score"],
        "history_years": history_years,
        "available_return": available_return,
        "available_cagr": available_cagr,
    }

    for y in HORIZONS:
        stats = trailing_stats(prices[t], y)
        row[f"return_{y}y"] = stats[0] if stats else np.nan
        row[f"cagr_{y}y"] = stats[1] if stats else np.nan
        row[f"excess_spy_{y}y"] = row[f"return_{y}y"] - benchmark.get(y, np.nan)

    rows.append(row)

returns = pd.DataFrame(rows).sort_values("return_10y", ascending=False, na_position="last")
display(returns.head(20))

## 4. Ten-year winners

This is the cleanest first answer to the historical question because every company shown below has a full trailing ten-year window. The `excess_spy_10y` column measures simple percentage-point excess return over SPY, not risk-adjusted alpha.

In [ ]:
top_10y = (
    returns.dropna(subset=["return_10y"])
    .sort_values("return_10y", ascending=False)
)

display(
    top_10y[
        ["company", "ticker", "tier", "non_obviousness",
         "return_10y", "cagr_10y", "excess_spy_10y"]
    ].head(20).style.format({
        "return_10y": "{:.1%}",
        "cagr_10y": "{:.1%}",
        "excess_spy_10y": "{:.1%}",
    })
)

fig = px.bar(
    top_10y.head(20).sort_values("return_10y"),
    x="return_10y",
    y="ticker",
    orientation="h",
    color="tier_name",
    hover_data=["company", "non_obviousness", "cagr_10y"],
    title="Top AI-linked stocks by trailing 10-year adjusted return",
)
fig.update_xaxes(tickformat=".0%")
fig.show()

## 5. Compare the three AI-economy layers

The median is especially useful because a few extraordinary winners can dominate the mean. These are descriptive group comparisons only: industry mix, geography, company age, and valuation differ across tiers.

In [ ]:
tier_summary = (
    returns.dropna(subset=["return_10y"])
    .groupby(["tier", "tier_name"])
    .agg(
        companies=("ticker", "count"),
        median_10y_return=("return_10y", "median"),
        mean_10y_return=("return_10y", "mean"),
        median_10y_cagr=("cagr_10y", "median"),
        median_excess_spy=("excess_spy_10y", "median"),
    )
    .reset_index()
)
display(tier_summary.style.format({
    "median_10y_return": "{:.1%}",
    "mean_10y_return": "{:.1%}",
    "median_10y_cagr": "{:.1%}",
    "median_excess_spy": "{:.1%}",
}))

long = returns.melt(
    id_vars=["company", "ticker", "tier_name"],
    value_vars=[f"return_{y}y" for y in HORIZONS],
    var_name="horizon",
    value_name="return",
).dropna()

fig = px.box(
    long, x="horizon", y="return", color="tier_name", points="all",
    hover_data=["company", "ticker"],
    title="Return distributions by AI-economy layer",
)
fig.update_yaxes(tickformat=".0%")
fig.show()

## 6. Hidden-beneficiary screen

Tier 3 is the distinctive part of this project. These companies participate in AI through **physical constraints**: electricity, grid capacity, cooling, construction, optics, or interconnects.

The qualitative `non_obviousness` field is only a starting hypothesis. A later stage should replace it with dated evidence from filings, earnings calls, contracts, and segment data.

In [ ]:
hidden = (
    returns.loc[returns["tier"].eq(3)]
    .sort_values(["return_10y", "available_cagr"], ascending=False, na_position="last")
)

display(
    hidden[
        ["company", "ticker", "primary_ai_link", "non_obviousness",
         "return_10y", "cagr_10y", "available_cagr"]
    ].style.format({
        "return_10y": "{:.1%}",
        "cagr_10y": "{:.1%}",
        "available_cagr": "{:.1%}",
    })
)

fig = px.scatter(
    returns.dropna(subset=["cagr_10y"]),
    x="non_obviousness_score",
    y="cagr_10y",
    color="tier_name",
    hover_name="company",
    hover_data=["ticker", "primary_ai_link"],
    title="10-year CAGR vs qualitative non-obviousness",
)
fig.update_xaxes(
    tickmode="array",
    tickvals=[1, 2, 3, 4],
    ticktext=["Low", "Medium", "High", "Very High"],
)
fig.update_yaxes(tickformat=".0%")
fig.show()

## 7. Growth of a hypothetical $10,000

To make the scale of the strongest decade-long returns intuitive, rebase the top ten full-history performers and SPY to $10,000 at the beginning of the common window.

In [ ]:
ten_year_start = prices.index.max() - pd.DateOffset(years=10)
plot_tickers = top_10y.head(10)["ticker"].tolist() + [BENCHMARK]

fig = go.Figure()
for t in plot_tickers:
    s = prices[t].dropna().loc[ten_year_start:]
    if s.empty:
        continue
    value = 10_000 * s / s.iloc[0]
    fig.add_trace(go.Scatter(x=value.index, y=value, mode="lines", name=t))

fig.update_layout(
    title="Growth of $10,000: top ten AI-linked performers vs SPY",
    xaxis_title="Date",
    yaxis_title="Hypothetical adjusted value ($)",
    hovermode="x unified",
)
fig.show()

## 8. Save the reproducible snapshot

The output is generated data; the hand-maintained `company_universe.csv` remains the source of truth for company classification.

In [ ]:
OUTPUT_DIR = UNIVERSE_PATH.parent.parent / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / "market_return_snapshot.csv"
returns.to_csv(output_path, index=False)
print(f"Saved {len(returns)} rows to {output_path}")

## Interpretation and next stages

This notebook **does not establish** that AI caused a company's historical stock performance. The universe was assembled in 2026, so survivorship and hindsight are major concerns.

The stronger version of the project should now add:

1. **Fundamentals:** revenue, margins, free cash flow, capex, valuation, and segment growth.
2. **AI-exposure evidence:** dated filings, earnings-call language, contracts, customer disclosures, and data-center revenue.
3. **Recognition dates:** estimate when each less-obvious company's AI exposure became economically visible.
4. **Event analysis:** compare returns and fundamentals before and after recognition, with market and sector benchmarks.
5. **Dependency graph:** model `models → compute → memory/networking → servers → facilities → cooling/power → grid/generation`.

The long-run goal is not to identify companies *after* they become famous AI winners. It is to test whether **bottlenecks in the AI dependency chain could have revealed them earlier**.